# 07 - Attribution case study: arguing one strategic-persuasion label by hand

Paper context: reviewers will ask whether the 'Why?' card's attribution can be *defended from the
transcript text*, not just produced by the machinery. This notebook extracts the real, executed
system output for every strategic_persuasion turn in the 150-transcript run, dumps the full
six-turn context around each, and scaffolds the human verification that becomes the worked case
study (paper S4/S6). Every number printed here comes from released checkpoints - nothing is
recomputed, nothing is invented.

In [1]:
from pathlib import Path
import json
import sys

root = Path.cwd()
while root != root.parent and not (root / 'src').is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src.utils import CheckpointManager

OUT_DIR = root / 'reports' / 'case_study'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT = CheckpointManager(str(root / 'checkpoints'))
print({'out_dir': str(OUT_DIR), 'checkpoint_keys_available': len(CKPT.list_keys())})

{'out_dir': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/case_study', 'checkpoint_keys_available': 314}


## 1. Locate the executed-run checkpoints and scan for strategic_persuasion turns

The full-corpus run stored one analysis checkpoint per transcript, content-addressed
(`analysis__<id>__<content_hash>__r0s0`, engine.py) and persisted as pickles. The bare
`analysis__<id>__r0s0` files are an earlier variant without the content hash and are NOT the
paper's run - we select only the 4-part (hash-suffixed) keys, which the paper pipeline's own
logs (notebooks/02) confirm were the executed caches. We load every cached analysis and collect
turns whose attribution label is strategic_persuasion. The paper reports exactly 3 such turns in
900 - this cell should confirm that count from the checkpoints themselves, which doubles as a
sanity check that the released caches match the paper.

In [2]:
PAPER_REPORTED_STRATEGIC_TURNS = 3

def is_paper_analysis_key(key: str) -> bool:
    # Paper run keys look like: analysis__<id>__<content_hash>__r0s0  (4 parts)
    # Older bare variant:        analysis__<id>__r0s0                  (3 parts)
    return key.startswith('analysis__') and len(key.split('__')) == 4

def collect_strategic_persuasion_turns():
    records = []
    for key in CKPT.list_keys():
        if not is_paper_analysis_key(key):
            continue
        try:
            result = CKPT.load(key, fmt='pkl')
        except Exception:
            continue  # not an analysis checkpoint (e.g. loader cache); skip, never guess
        if not isinstance(result, dict) or 'attribution' not in result:
            continue
        for i, attr in enumerate(result['attribution']):
            if attr and attr.get('label') == 'strategic_persuasion':
                records.append({
                    'checkpoint_key': key,
                    'transcript_id': result.get('transcript_id'),
                    'topic': result.get('topic'),
                    'turn_index': i,
                    'speaker': result['speakers'][i],
                    'text': result['texts'][i],
                    'rhetoric': result['rhetoric'][i],
                    'prev_rhetoric': result['rhetoric'][i - 1] if i else None,
                    'quality': result['quality'][i],
                    'stance_S': result['stance'][i]['S'],
                    'prev_stance_S': result['stance'][i - 1]['S'] if i else None,
                    'signals': attr.get('signals'),
                    'confidence': attr.get('confidence'),
                })
    return records

strategic_turns = collect_strategic_persuasion_turns()
print({'strategic_persuasion_turns_found': len(strategic_turns),
       'paper_reported': PAPER_REPORTED_STRATEGIC_TURNS,
       'match': len(strategic_turns) == PAPER_REPORTED_STRATEGIC_TURNS})
if len(strategic_turns) == 0:
    raise RuntimeError('No strategic_persuasion turns found in hash-suffixed analysis '
                       'checkpoints - check the variant filter before proceeding.')
if len(strategic_turns) != PAPER_REPORTED_STRATEGIC_TURNS:
    print('WARNING: count differs from the paper-reported number - investigate before '
          'citing this run in the paper; do NOT adjust the paper to match silently.')
for rec in strategic_turns:
    print('-' * 60)
    print(rec['transcript_id'], '|', rec['topic'], '| turn', rec['turn_index'],
          '| speaker', rec['speaker'], '| conf', round(rec['confidence'] or 0.0, 3))

{'strategic_persuasion_turns_found': 3, 'paper_reported': 3, 'match': True}
------------------------------------------------------------
125.0 | Should Governments Have the Right to Censor the Internet? | turn 3 | speaker 125.0_pro | conf 0.459
------------------------------------------------------------
176.0 | Is Government Surveillance Necessary for National Security? | turn 2 | speaker 176.0_con | conf 0.311
------------------------------------------------------------
303.0 | Should the Death Penalty Be Legal? | turn 2 | speaker 303.0_con | conf 0.61


## 2. Dump full six-turn context per flagged turn

For each flagged turn we export the complete transcript (all six turns, speaker + text + per-turn
rhetoric + stance trajectory + attribution labels) so the case-study argument can quote adjacent
turns verbatim: the persuasive shift is defined by the *previous* turn's rhetoric distribution
(causal/empirical) versus the flagged turn's (emotional/moral), and by the flagged speaker's
subsequent stance movement.

In [3]:
def dump_context(rec: dict) -> dict:
    result = CKPT.load(rec['checkpoint_key'], fmt='pkl')
    turns = []
    for i in range(len(result['texts'])):
        turns.append({
            'turn': i,
            'speaker': result['speakers'][i],
            'text': result['texts'][i],
            'rhetoric': result['rhetoric'][i]['probs'] if result['rhetoric'][i] else None,
            'rhetoric_label': result['rhetoric'][i]['label'] if result['rhetoric'][i] else None,
            'quality': result['quality'][i],
            'stance_S': result['stance'][i]['S'],
            'attribution_label': (result['attribution'][i] or {}).get('label'),
            'attribution_signals': (result['attribution'][i] or {}).get('signals'),
        })
    return {'transcript_id': rec['transcript_id'], 'topic': rec['topic'],
            'flagged_turn': rec['turn_index'], 'turns': turns}

contexts = [dump_context(rec) for rec in strategic_turns]
out_path = OUT_DIR / 'strategic_persuasion_contexts.json'
out_path.write_text(json.dumps(contexts, indent=2, ensure_ascii=False))
print({'exported': str(out_path), 'n_transcripts': len(contexts)})
for ctx in contexts:
    print('=' * 70)
    print(ctx['transcript_id'], '|', ctx['topic'])
    for t in ctx['turns']:
        marker = ' >>> FLAGGED' if t['turn'] == ctx['flagged_turn'] else ''
        print(f"[{t['turn']}] {t['speaker']} | {t['rhetoric_label']} | "
              f"q={t['quality']:.2f} S={t['stance_S']:+.3f} "
              f"label={t['attribution_label']}{marker}")
        print('    ', (t['text'] or '')[:180].replace('\n', ' '))

{'exported': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/case_study/strategic_persuasion_contexts.json', 'n_transcripts': 3}
125.0 | Should Governments Have the Right to Censor the Internet?
[0] 125.0_con | causal | q=0.25 S=+0.100 label=no_inflection
     The United States has governing parties that control on the town level, the state level, and on the national level. Laws are in some ways based on geographic location--laws are cre
[1] 125.0_pro | causal | q=0.25 S=+0.100 label=echo
     The government should have the right to remove sensitive pieces of information that are illegal or confidential to stop people from having access to them. People keep leaking milit
[2] 125.0_con | causal | q=0.25 S=+0.190 label=no_inflection
     The US government has no ownership over online domains. Posting information to the World of Tanks forum is akin to having a conversation in the middle of the ocean--there is no jur
[3] 125.0_pro | moral | q=0.75 S=+0.374 label=stra

## 3. Human verification scaffold (fills the paper's worked case study)

For each flagged turn, answer the four questions below IN PROSE after reading the dumped context.
These answers, checked against the card's signals, are what a skeptical reviewer needs: they show
the attribution is defensible (or refutable) from the transcript itself, independent of the system.

Checklist per flagged turn:
1. **Rhetorical shift**: does the *previous* turn by the other speaker read as causal/empirical
   (data, mechanisms) while the flagged turn reads as emotional/moral (values, affect)? Quote it.
2. **Stance movement**: did the flagged speaker's stance move substantially after this turn
   (compare `prev_stance_S` to later-turn `stance_S` for the same speaker)?
3. **Alternative explanations**: is the shift better explained by evidence adoption (new substantive
   content), echo (mirroring the other speaker's wording), or anchoring (restating own position)?
4. **Verdict**: does the human reader endorse, refute, or call ambiguous the system's
   strategic_persuasion label - and why, in two sentences?

Write the final case-study paragraph (150-250 words) in `reports/case_study/case_study.md`.
The `verification` fields below are the structured record; keep them in sync with the prose.

In [4]:
VERIFICATION_TEMPLATE = {
    'transcript_id': None,
    'flagged_turn': None,
    'shift_quote_prev_turn': '',   # verbatim quote supporting causal/empirical -> emotional/moral
    'shift_quote_flagged_turn': '',
    'stance_before': None,
    'stance_after': None,
    'alternative_explanations_considered': [],
    'human_verdict': '',           # endorse | refute | ambiguous
    'verdict_rationale': '',
}

verification = [dict(VERIFICATION_TEMPLATE,
                     transcript_id=ctx['transcript_id'],
                     flagged_turn=ctx['flagged_turn']) for ctx in contexts]
verify_path = OUT_DIR / 'verification_scaffold.json'
if not verify_path.exists():
    verify_path.write_text(json.dumps(verification, indent=2, ensure_ascii=False))
    print({'scaffold_written': str(verify_path),
           'next_step': 'fill fields after reading each dumped context, then write case_study.md'})
else:
    print({'status': 'scaffold_exists', 'path': str(verify_path),
       'note': 'not overwriting - fill in and keep'})

{'scaffold_written': '/home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/case_study/verification_scaffold.json', 'next_step': 'fill fields after reading each dumped context, then write case_study.md'}


In [5]:
"""
Fill reports/case_study/verification_scaffold.json and write reports/case_study/case_study.md
from the dumped strategic-persuasion contexts (run after 07_attribution_case_study.ipynb).

Grounding rule: every quote and stance number is verified against
strategic_persuasion_contexts.json BEFORE anything is written; any mismatch raises
and exits without writing. Idempotent: already-filled scaffold records are never clobbered.
"""
import json, os, re, sys
from pathlib import Path

# ---------------------------------------------------------------- root + inputs
def find_root() -> Path:
    root = Path.cwd()
    while root != root.parent and not (root / "reports" / "case_study").is_dir():
        root = root.parent
    if not (root / "reports" / "case_study").is_dir():
        sys.exit("Could not locate reports/case_study/ - run from inside the repo.")
    return root

ROOT = find_root()
OUT = ROOT / "reports" / "case_study"
CTX_PATH = OUT / "strategic_persuasion_contexts.json"
SCAFFOLD_PATH = OUT / "verification_scaffold.json"
MD_PATH = OUT / "case_study.md"

if not CTX_PATH.exists():
    sys.exit(f"{CTX_PATH} not found - run notebook 07 first.")
ctxs = json.loads(CTX_PATH.read_text(encoding="utf-8"))
by_id = {c["transcript_id"]: c for c in ctxs}

def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

# ---------------------------------------------------------------- the records (provisional analyst readings)
ANALYST_NOTE = ("First-pass analyst reading by the paper-assistant agent against the dumped "
                "context; verify against the full transcript before citing as an authorial judgment.")

RECORDS = [
    dict(
        transcript_id="303.0", flagged_turn=2,
        shift_quote_prev_turn=("For example, if someone murders 10+ people, I don't think it's "
            "acceptable to give that criminal a life that they get to live in prison while the "
            "victims' never get to receive that same opportunity."),
        shift_quote_flagged_turn=("What if, which is not uncommon, later evidence suggests the "
            "criminal was indeed innocent but no longer alive, can that be true justice? A life in "
            "prison demands those which are in fact guilty to dwell on their crimes and require "
            "them to spend countless days regretting their decision."),
        stance_before=0.291, stance_after=0.685,
        alternative_explanations_considered=[
            "evidence_adoption: no new data or mechanism introduced; the turn poses moral hypotheticals only",
            "echo: wording diverges from the pro turn rather than mirroring it",
            "anchoring: the moral risk-of-error frame is new in this debate, not a restatement of an earlier position",
        ],
        human_verdict="endorse",
        verdict_rationale=("Textbook rhetorical reframe: the pro's empirical case is answered with a "
            "values argument (wrongful-conviction risk) with no new evidence and no mirroring; the pro's "
            "next turn concedes the sentiment and the final turn adopts the uncertainty standard outright. "
            "Note the system labels that final turn echo while a human reads concession - itself a "
            "demonstrative gap between candidate label and human reading."),
        supporting_quotes=[
            {"turn": 3, "role": "opponent concession in the next turn",
             "quote": "While I agree with the sentiment of someone having to live with their actions"},
            {"turn": 5, "role": "opponent adopts the uncertainty standard in the final turn (system labels this turn echo)",
             "quote": ("I think it needs to be proven with the upmost certainty that the criminal is "
                       "guilty with strong evidence before we take their life.")},
        ],
    ),
    dict(
        transcript_id="176.0", flagged_turn=2,
        shift_quote_prev_turn=("Just as parents protect and look after their children, the government "
            "has the right to look after us to ensure our safety and security."),
        shift_quote_flagged_turn=("National security is extremely important. However, for citizens like "
            "myself who have a lifetime history of no terrorist involvement, history of a criminal "
            "record, we should be given our right to privacy."),
        stance_before=0.291, stance_after=0.564,
        alternative_explanations_considered=[
            "accommodation: the turn opens by conceding the opponent's premise ('National security is extremely important. However, ...'), so accord is as defensible a reading as strategy",
            "echo: mirrors the opponent's opening phrase but below threshold",
            "evidence_adoption: none; the turn challenges the opponent's evidence ('where is the proof of that?') rather than adding new support",
        ],
        human_verdict="ambiguous",
        verdict_rationale=("Real affective pivot, and the full text shows a substantive rebuttal (inverted "
            "parent analogy, evidence challenge), which strengthens the strategic reading - but the opening "
            "concession stands, so accommodation is equally defensible. The card correctly reports the "
            "fired signal without adjudicating between strategy and accommodation."),
        supporting_quotes=[
            {"turn": 2, "role": "inverted parent analogy within the flagged turn",
             "quote": ("As a parent, I would not listen in on my child's private conversations. "
                       "The government is certainly not my parent.")},
            {"turn": 2, "role": "evidence challenge within the flagged turn",
             "quote": "where is the proof of that?"},
        ],
    ),
    dict(
        transcript_id="125.0", flagged_turn=3,
        shift_quote_prev_turn=("The US government has no ownership over online domains. Posting "
            "information to the World of Tanks forum is akin to having a conversation in the middle of "
            "the ocean--there is no jurisdiction that should control the conversation."),
        shift_quote_flagged_turn=("The government has ownership of certain types of information "
            "regardless of physical location. They may not have the right to censor anything they want, "
            "but they are allowed to censor pieces of information that are considered government property."),
        stance_before=0.100, stance_after=0.434,
        alternative_explanations_considered=[
            "rhetoric-annotation question: the turn is an ownership/property-rights rebuttal; a human might categorize it as legal/causal rather than pure moral (1.0), so the computed shift magnitude partly rides on the annotator's categorization",
            "baseline artifact: the previous turn has a uniform rhetoric distribution (0.25 on all four dimensions), a maximally uninformative baseline that inflates the computed shift",
            "anchoring/evidence_adoption/echo: all below thresholds; the pro's own next turn is itself labeled echo",
        ],
        human_verdict="ambiguous",
        verdict_rationale=("A moral counter-argument it is, but evidence that it was strategic persuasion "
            "is thin: uniform-baseline shift, contestable pure-moral categorization of a property-rights "
            "argument, and subsequent own-turn labeled echo. Lean refute-as-strategic; recorded as "
            "ambiguous per the scaffold's coarse verdict set."),
        supporting_quotes=[
            {"turn": 5, "role": "flagged speaker's next turn is itself labeled echo",
             "quote": "The government owns the right to the information that is being shared."},
        ],
    ),
]

# ---------------------------------------------------------------- verification gate (no fabrication)
errors = []
for rec in RECORDS:
    ctx = by_id.get(rec["transcript_id"])
    if ctx is None:
        errors.append(f"{rec['transcript_id']}: not present in dumped contexts"); continue
    turns = {t["turn"]: t for t in ctx["turns"]}
    flag = ctx["flagged_turn"]
    if flag != rec["flagged_turn"]:
        errors.append(f"{rec['transcript_id']}: flagged_turn mismatch (dump {flag} vs record {rec['flagged_turn']})")
    for field, ti in [("shift_quote_prev_turn", flag - 1), ("shift_quote_flagged_turn", flag)]:
        if ti >= 0 and norm(rec[field]) not in norm(turns[ti]["text"]):
            errors.append(f"{rec['transcript_id']} {field}: quote not verbatim in dumped turn {ti}")
    for q in rec["supporting_quotes"]:
        if norm(q["quote"]) not in norm(turns[q["turn"]]["text"]):
            errors.append(f"{rec['transcript_id']} supporting quote (turn {q['turn']}): not verbatim")
    speaker = turns[flag]["speaker"]
    own = [t for t in ctx["turns"] if t["speaker"] == speaker]
    prev_own = [t for t in own if t["turn"] < flag]
    before = (prev_own[-1] if prev_own else turns[flag - 1])["stance_S"]
    after = own[-1]["stance_S"]
    if abs(round(before, 3) - rec["stance_before"]) > 0.001:
        errors.append(f"{rec['transcript_id']}: stance_before {rec['stance_before']} != checkpoint {round(before, 3)}")
    if abs(round(after, 3) - rec["stance_after"]) > 0.001:
        errors.append(f"{rec['transcript_id']}: stance_after {rec['stance_after']} != checkpoint {round(after, 3)}")

if errors:
    print("GROUNDING CHECK FAILED - nothing written:")
    for e in errors:
        print("  -", e)
    sys.exit(1)
print("Grounding check passed: all quotes verbatim, all stance numbers match the checkpoints.")

# ---------------------------------------------------------------- merge into scaffold (never clobber filled records)
scaffold = json.loads(SCAFFOLD_PATH.read_text(encoding="utf-8")) if SCAFFOLD_PATH.exists() else []
existing = {r.get("transcript_id"): i for i, r in enumerate(scaffold)}
filled, kept = 0, 0
for rec in RECORDS:
    entry = dict(rec)
    entry["provisional"] = True
    entry["analyst_note"] = ANALYST_NOTE
    idx = existing.get(rec["transcript_id"])
    if idx is not None and scaffold[idx].get("human_verdict") not in ("", None):
        print(f"  kept existing filled record for {rec['transcript_id']} (not clobbered)")
        kept += 1
        continue
    entry = {k: entry[k] for k in ["transcript_id", "flagged_turn", "shift_quote_prev_turn",
             "shift_quote_flagged_turn", "stance_before", "stance_after",
             "alternative_explanations_considered", "human_verdict", "verdict_rationale",
             "supporting_quotes", "provisional", "analyst_note"]}
    if idx is not None:
        scaffold[idx] = entry
    else:
        scaffold.append(entry)
    filled += 1

tmp = SCAFFOLD_PATH.with_suffix(".json.tmp")
tmp.write_text(json.dumps(scaffold, indent=2, ensure_ascii=False), encoding="utf-8")
os.replace(tmp, SCAFFOLD_PATH)
print(f"Wrote {SCAFFOLD_PATH}  ({filled} records filled, {kept} kept)")

# ---------------------------------------------------------------- case_study.md (upgrade only; never downgrade)
CASE_STUDY_MD = """# Worked case study: the three `strategic_persuasion` attributions

Source: `strategic_persuasion_contexts.json` (dumped verbatim from the released
hash-suffixed `analysis__*` checkpoints by `notebooks/experiments/07_attribution_case_study.ipynb`).
Counts and confidences below are read directly from the checkpoints; the per-case readings are one
analyst's provisional judgments made against the four-question scaffold — they are arguments from
the transcript text, not measurements, and are meant to be checked (and if necessary revised) by the
authors before being cited.

Cross-case fact worth noting: the system's (uncalibrated) confidence ordering is
303.0 (0.610) > 125.0 (0.459) > 176.0 (0.311). An informal human readability ordering agrees that
303.0 is the most defensible case, but ranks 176.0 above 125.0 — i.e., the confidence score does not
fully track an informal human ordering. This is consistent with the paper's treatment of the
confidence display as an uncalibrated heuristic (§5, H6) and is exactly why the card surfaces
signals + alternatives rather than a verdict.

**Scaffold status:** `verification_scaffold.json` is filled with these same provisional readings
(each record marked `"provisional": true`), kept in sync with the prose below. The authors should
still verify each case against the full transcript before citing.

---

## Case 1 — transcript 303.0, death penalty, turn 2 (con), confidence 0.610 — most defensible

**Card:** `strategic_persuasion` (only this signal fired; evidence adoption, echo, anchoring all
false). Rhetorical shift: previous turn predominantly empirical (0.36), flagged turn pure moral (1.0).

**1. Rhetorical shift (quoted).**
- Prev (pro, empirical framing): *"I believe that capital punishment is a fair assessment/ruling
  when it fits the criminal's crime. For example, if someone murders 10+ people, I don't think it's
  acceptable to give that criminal a life that they get to live…"*
- Flagged (con, pure moral): *"Is it fair in all circumstances though? What if, which is not
  uncommon, later evidence suggests the criminal was indeed innocent but no longer alive, can that
  be true justice? A life in prison demands those which are in fact guilty to dwell on their crimes…"*

**2. Stance movement.** The flagged speaker's own trajectory strengthens monotonically after the
moral reframe: 0.291 → 0.521 → 0.685 (turns 0, 2, 4).

**3. Alternative explanations considered.** Evidence adoption: no new data or mechanism is
introduced — the turn poses moral hypotheticals. Echo: wording diverges from the pro turn rather
than mirroring it. Anchoring: the con is not restating an earlier position; the moral-risk-of-error
frame is new in this debate.

**4. Provisional verdict: defensible (lean endorse).** The turn is a textbook rhetorical reframe —
the opponent's empirical case is met with a values argument, not evidence — and the opponent's next
turn visibly accommodates it: *"While I agree with the sentiment of someone having to live with
their actions…"*. A human reader can verify each of these steps from the dumped context.

**Full-text note (complete context reviewed).** The arc is even stronger than the snippet
suggested: the pro's *final* turn (5) concedes the con's uncertainty standard outright — *"I don't
think the death penalty should be used in circumstances such as this scenario, I think it needs to
be proven with the upmost certainty that the criminal is guilty with strong evidence before we
take their life."* — while the system labels that same turn `echo`. A human reads concession; the
signal fires on mirrored structure. That gap is itself demonstrative: the card is a candidate
explanation to be checked against the transcript, not a verdict.

## Case 2 — transcript 176.0, surveillance, turn 2 (con), confidence 0.311 — contestable, informative

**Card:** `strategic_persuasion` (only signal fired). Shift: causal-heavy prev turn → emotional/moral
(0.5/0.5) flagged turn.

**1. Rhetorical shift (quoted).**
- Prev (pro, causal): *"National security is of utmost importance. With the rise of terrorist
  attacks, we need to be sure we can be kept safe…"*
- Flagged (con, emotional/moral): *"National security is extremely important. However, for citizens
  like myself who have a lifetime history of no terrorist involvement, history of a criminal record,
  we should be given our right to privacy."*

**2. Stance movement.** Con: 0.291 → 0.521 → 0.564 — strengthens, but with diminishing increments.

**3. Alternative explanations considered.** The flagged turn *partially concedes the opponent's
premise* ("National security is extremely important. However, …"), so accommodation/accord is a
plausible alternative reading that the card does not exclude. Echo: partial mirroring of the
opponent's opening phrase, but below the signal threshold. Evidence adoption: none.

**4. Provisional verdict: ambiguous.** The affective pivot is real, but the concession framing makes
accommodation equally defensible. This case is valuable precisely because it shows what the system
does *not* establish: the card reports that the strategic-persuasion signal fired and that no
alternative signal did — it does not, and should not, adjudicate between strategy and accommodation.

**Full-text note.** The complete turn also challenges the opponent's evidence (*"where is the
proof of that?"*) and inverts the parent analogy (*"As a parent, I would not listen in on my
child's private conversations. The government is certainly not my parent."*), which makes the
strategic reading more substantive than the snippet alone suggested — but the opening concession
stands, so the verdict remains ambiguous.

## Case 3 — transcript 125.0, internet censorship, turn 3 (pro), confidence 0.459 — weakest

**Card:** `strategic_persuasion` (only signal fired). Shift: uniform baseline (0.25 on all four
rhetoric dimensions) → pure moral (1.0).

**1. Rhetorical shift (quoted).**
- Prev (con, uniform baseline): *"The US government has no ownership over online domains. Posting
  information to the World of Tanks forum is akin to having a conversation in the middle of the
  ocean…"*
- Flagged (pro, pure moral): *"The government has ownership of certain types of information
  regardless of physical location. They may not have the right to censor anything they want, but
  they are allowed to censor pieces of information that are considered government property."*

**2. Stance movement.** Pro: 0.374 (flagged turn) → 0.434 (turn 5) — small; and the pro's turn 5 is
itself labeled `echo`, further weakening a strategic reading of the subsequent movement.

**3. Alternative explanations considered.** The shift magnitude is computed against a *maximally
uninformative* previous-turn baseline (equal mass on all four dimensions), so the 0→1 moral jump
overstates how deliberate the pivot looks to a human reader. Echo/anchoring/evidence: all below
their thresholds.

**4. Provisional verdict: ambiguous (lean refute-as-strategic).** A moral counter-argument it is;
evidence that it was *strategic* persuasion is thin. Reported honestly, this is the case where the
signal most likely over-fires — and it is useful to show reviewers that the workbench's artifacts
let a reader reach exactly that conclusion.

**Full-text note.** The flagged turn is a direct ownership/property-rights rebuttal (*"The
government has ownership of certain types of information regardless of physical location"*); a
human reader might categorize it as legal/causal rather than pure moral (1.0), so the computed
shift magnitude partly rides on the rhetoric annotator's categorization — a second, independent
reason to distrust the strength of this flag.

---

## What this case study establishes (and does not)

- **Establishes:** every flagged attribution is inspectable down to the transcript text, the
  per-turn rhetoric distribution, and the signal vector; a reader can argue for or against each
  label without re-running the system.
- **Does not establish:** that the strategic-persuasion signal is accurate in general (n = 3, no
  statistical claim intended), or that confidence is calibrated (H6 already reports it is not).
- **Framing for the paper:** consistent with the evaluation decision that *ambiguous is a legitimate
  system output* — the workbench makes candidate explanations and their evidence inspectable; it
  does not certify causes.
"""

if MD_PATH.exists() and "Full-text note (complete context reviewed)" in MD_PATH.read_text(encoding="utf-8"):
    print(f"Kept existing {MD_PATH} (already the final version)")
else:
    MD_PATH.write_text(CASE_STUDY_MD, encoding="utf-8")
    print(f"Wrote {MD_PATH}")

print("\nSummary (provisional verdicts, pending author verification):")
for rec in RECORDS:
    print(f"  {rec['transcript_id']}  flagged turn {rec['flagged_turn']}  ->  {rec['human_verdict']}")

Grounding check passed: all quotes verbatim, all stance numbers match the checkpoints.
Wrote /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/case_study/verification_scaffold.json  (3 records filled, 0 kept)
Wrote /home/pakdd/github repo/EACL DEMO 2027/belief_debate_analyzer/reports/case_study/case_study.md

Summary (provisional verdicts, pending author verification):
  303.0  flagged turn 2  ->  endorse
  176.0  flagged turn 2  ->  ambiguous
  125.0  flagged turn 3  ->  ambiguous
